In [16]:
import os

In [17]:
%pwd

'c:\\Users\\amrut\\Downloads\\kidney-Disease-Classifiaction-DLProject'

In [5]:
os.chdir("../")

In [6]:
%pwd

'c:\\Users\\amrut\\Downloads\\kidney-Disease-Classifiaction-DLProject'

In [ ]:
os.environ["MLFLOW_TRACKING_URI"] = "https://dagshub.com/AmruthGowda34/kidney-Disease-Classifiaction-DLProject.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"] = "AmruthGowda34"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "e7b944bf1fb1cb6addfd587260d5e3a1ec0b9b49"

In [20]:
import tensorflow as tf

In [21]:
model = tf.keras.models.load_model("artifacts/training/model_v2.h5")

In [23]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class EvaluationConfig:
    path_of_model: Path
    training_data: Path
    all_params: dict
    mlflow_uri: str
    params_image_size: list
    params_batch_size: int

In [24]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories, save_json

In [25]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    def get_evaluation_config(self) -> EvaluationConfig:
        eval_config = EvaluationConfig(
            path_of_model="artifacts/training/model_v2.h5",
            training_data=(
                "artifacts/data_ingestion/"
                "CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone/"
                "CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone"
            ),
            mlflow_uri=(
                "https://dagshub.com/AmruthGowda34/"
                "kidney-Disease-Classifiaction-DLProject.mlflow"
            ),
            all_params=self.params,
            params_image_size=self.params.IMAGE_SIZE,
            params_batch_size=self.params.BATCH_SIZE
        )
        return eval_config

In [27]:
import tensorflow as tf
from pathlib import Path
import mlflow
import mlflow.keras
from urllib.parse import urlparse

In [28]:
class Evaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config

    def _valid_generator(self):
        # Use the same preprocessing as the VGG16 training pipeline.
        datagenerator_kwargs = dict(
            preprocessing_function=(
                tf.keras.applications.vgg16.preprocess_input
            ),
            validation_split=0.30
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = (
            tf.keras.preprocessing.image.ImageDataGenerator(
                **datagenerator_kwargs
            )
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=str(self.config.training_data),
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )

    @staticmethod
    def load_model(path: Path) -> tf.keras.Model:
        return tf.keras.models.load_model(path)

    def evaluation(self):
        self.model = self.load_model(self.config.path_of_model)

        self._valid_generator()

        self.score = self.model.evaluate(
            self.valid_generator,
            verbose=1
        )

        self.save_score()

    def save_score(self):
        scores = {
            "loss": float(self.score[0]),
            "accuracy": float(self.score[1])
        }

        save_json(
            path=Path("scores.json"),
            data=scores
        )

    def log_into_mlflow(self):
        # Configure the remote MLflow tracking server.
        mlflow.set_tracking_uri(self.config.mlflow_uri)

        tracking_url_type_store = urlparse(
            mlflow.get_tracking_uri()
        ).scheme

        with mlflow.start_run():
            mlflow.log_params(dict(self.config.all_params))

            mlflow.log_metrics({
                "loss": float(self.score[0]),
                "accuracy": float(self.score[1])
            })

            if tracking_url_type_store != "file":
                mlflow.keras.log_model(
                    self.model,
                    artifact_path="model",
                    registered_model_name="VGG16Model"
                )
            else:
                mlflow.keras.log_model(
                    self.model,
                    artifact_path="model"
                )


In [29]:
try:
    config = ConfigurationManager()
    eval_config = config.get_evaluation_config()

    evaluation = Evaluation(config=eval_config)
    evaluation.evaluation()
    evaluation.log_into_mlflow()

except Exception as e: 
    raise e

[2026-10-19 17:35:49,167: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-10-19 17:35:49,174: INFO: common: yaml file: params.yaml loaded successfully]
[2026-10-19 17:35:49,180: INFO: common: created directory at: artifacts]
Found 3732 images belonging to 4 classes.
234/234 [==============================] - 671s 3s/step - loss: 215.0302 - accuracy: 0.7787
[2026-10-19 17:47:01,810: INFO: common: json file saved at: scores.json]


2026/10/19 17:47:04 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


[2026-10-19 17:47:06,251: WARNING: save: Found untraced functions such as _jit_compiled_convolution_op, _jit_compiled_convolution_op, _jit_compiled_convolution_op, _jit_compiled_convolution_op, _jit_compiled_convolution_op while saving (showing 5 of 14). These functions will not be directly callable after loading.]
INFO:tensorflow:Assets written to: C:\Users\amrut\AppData\Local\Temp\tmplrght3_3\model\data\model\assets
[2026-10-19 17:47:07,130: INFO: builder_impl: Assets written to: C:\Users\amrut\AppData\Local\Temp\tmplrght3_3\model\data\model\assets]


2026/10/19 17:47:26 WARNING mlflow.utils.requirements_utils: The following packages were not found in the public PyPI package index as of 2023-02-28; if these packages are not present in the public PyPI index, you must install them manually before loading your model: {'backports-tarfile'}
Registered model 'VGG16Model' already exists. Creating a new version of this model...
2026/10/19 17:48:56 INFO mlflow.tracking._model_registry.client: Waiting up to 300 seconds for model version to finish creation.                     Model name: VGG16Model, version 2
Created version '2' of model 'VGG16Model'.


In [3]:
import os
import mlflow

# Set your DagsHub credentials and tracking URI
os.environ["MLFLOW_TRACKING_URI"] = (
    "https://dagshub.com/AmruthGowda34/"
    "kidney-Disease-Classifiaction-DLProject.mlflow"
)

os.environ["MLFLOW_TRACKING_USERNAME"] = "AmruthGowda34"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "e7b944bf1fb1cb6addfd587260d5e3a1ec0b9b49"

# Configure MLflow
mlflow.set_tracking_uri(
    os.environ["MLFLOW_TRACKING_URI"]
)

# Test experiment logging
mlflow.set_experiment("kidney-disease-test")

with mlflow.start_run():
    mlflow.log_param("test", "connection")
    mlflow.log_metric("accuracy", 0.73)

print("MLflow logging successful!")

2026/10/21 18:21:03 INFO mlflow.tracking.fluent: Experiment with name 'kidney-disease-test' does not exist. Creating a new experiment.


MLflow logging successful!
